In [1]:
import sys
import time
import numpy as np
import torch
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from sentence_transformers import CrossEncoder
from my_rag import search as s
from my_rag.evaluate import load_tests
from my_rag.config import MODELS_DIR

positives, refuse, faq_out = load_tests()

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", cache_folder=str(MODELS_DIR), device=device, max_length=512)
if device != "cpu":
    try:
        reranker.model.half()
    except Exception:
        pass
print(device, len(positives))

c:\projects\HotelAI-Intelligent-Agent_Final_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 21:50:27.370000 21668 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 30015.79it/s]
The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 393/393 [00:01<00:00, 380.06it/s]


cuda 105


In [5]:
def rerank_search(question, top_k=5, n=20):
    q = s._encode(question)
    if s.is_outside(question, q):
        return []
    best, dense = s.hybrid_rank(question, n=n, q=q)
    pairs = []
    for i in best:
        pairs.append((question, s._chunks[i]["text"]))
    scores = reranker.predict(pairs)
    order = np.argsort(-scores)
    results = []
    seen = set()
    for j in order:
        i = best[j]
        keys = s._answer_keys(s._chunks[i])
        if any(k in seen for k in keys):
            continue
        seen.update(keys)
        results.append({"text": s._chunks[i]["text"], "source": s._chunks[i]["source"], "score": round(float(scores[j]), 4)})
        if len(results) == top_k:
            break
    return results


def measure(fn, **kw):
    hits = 0
    times = []
    for item in positives:
        start = time.time()
        res = fn(item["question"], **kw)
        times.append(time.time() - start)
        top5 = []
        for r in res:
            top5.append(r["source"])
        if any(d in top5 for d in item["docs"]):
            hits += 1
    return round(100 * hits / len(positives)), round(sum(times) / len(times), 3), round(max(times), 3)


for r in rerank_search("عايز دبل لتلاتة، أنا ومراتي وابننا 5 سنين"):
    print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

print("without reranker      :", measure(s.search))
print("reranker on top 10    :", measure(rerank_search, n=10))
print("reranker on top 20    :", measure(rerank_search, n=20))

    0.0006 room_types.txt | Q: عايز أوضة تاخد 3 أفراد A: الديلوكس بتاخد 3 أفراد بـ 12,00
    0.0004 hotel_information.txt | Q: عايز أحجز لمراتي في فندق خمس نجوم، انتوا كده؟ A: الفندق خ
    0.0002 room_policies.txt | Q: إحنا اتنين كبار وطفلين، أنسب أوضة إيه؟ A: أربعة أفراد بال
    0.0002 booking_policy.txt | Q: إحنا 3 أفراد، ينفع نحجز أوضة واحدة؟ A: عدد الأفراد لازم ي
    0.0002 room_types.txt | Q: مساحة الأوضة الديلوكس كام متر؟ A: المساحات: السينجل 28 مت
without reranker      : (88, 0.017, 0.025)
reranker on top 10    : (90, 0.082, 0.274)
reranker on top 20    : (92, 0.089, 0.152)


In [6]:
test_pairs = [
    ("what time is check-in?", "Check-in time is 3:00 PM."),
    ("what time is check-in?", "The swimming pool closes at 10 PM."),
    ("الشيك إن الساعة كام؟", "Check-in time is 3:00 PM."),
]
print("fp16:", reranker.predict(test_pairs))

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", cache_folder=str(MODELS_DIR), device=device, max_length=512)
print("fp32:", reranker.predict(test_pairs))

for r in rerank_search("عايز دبل لتلاتة، أنا ومراتي وابننا 5 سنين"):
    print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

print("reranker fp32 top 20  :", measure(rerank_search, n=20))

fp16: [9.9486035e-01 2.0825010e-04 2.5183168e-01]


Loading weights: 100%|██████████| 393/393 [00:01<00:00, 335.93it/s]


fp32: [9.9485493e-01 2.0759786e-04 2.4886671e-01]
    0.0006 room_types.txt | Q: عايز أوضة تاخد 3 أفراد A: الديلوكس بتاخد 3 أفراد بـ 12,00
    0.0004 hotel_information.txt | Q: عايز أحجز لمراتي في فندق خمس نجوم، انتوا كده؟ A: الفندق خ
    0.0002 room_policies.txt | Q: إحنا اتنين كبار وطفلين، أنسب أوضة إيه؟ A: أربعة أفراد بال
    0.0002 booking_policy.txt | Q: إحنا 3 أفراد، ينفع نحجز أوضة واحدة؟ A: عدد الأفراد لازم ي
    0.0002 room_types.txt | Q: مساحة الأوضة الديلوكس كام متر؟ A: المساحات: السينجل 28 مت
reranker fp32 top 20  : (92, 0.234, 0.454)


## Reranker result (bge-reranker-v2-m3)
| setup | Recall@5 | avg time | max time |
|---|---|---|---|
| no reranker | 88% | 0.017s | 0.025s |
| reranker top 10 (fp16) | 90% | 0.082s | 0.274s |
| reranker top 20 (fp16) | 92% | 0.089s | 0.152s |
| reranker top 20 (fp32) | 92% | 0.234s | 0.454s |

Decision: not used. Gain is +4 (rule was +5), 5x slower, weak on Egyptian Arabic (best score 0.0006).